In [ ]:
!pip install -qqq 'pip' 'crewai[tools]==0.28.8' 'duckduckgo-search==5.3.0' 'langchain-groq==0.1.3' --progress-bar off

In [ ]:
import os
import re
from datetime import datetime
import pandas as pd
import requests
from crewai import Agent, Crew, Process, Task
from crewai_tools import tool
from google.colab import userdata
from langchain.agents import load_tools
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_groq import ChatGroq

def format_response(response: str) -> str:
  entries=re.split(r"(?<=]), (?=\[)", response)
  return [entry.strip("[]") for entry in entries]

os.environ ["GROQ_API_KEY"] = userdata.get("lama3")

In [ ]:
search_tool = DuckDuckGoSearchResults(backend="news",num_results=10)

In [ ]:
# Test search tool
response = search_tool.run("Agriculture news")
formatted_response = format_response(response)
print(formatted_response)

['snippet: A Minnesota dairy herd is quarantined after several cows became ill with the bird flu. An agriculture specialist from the University of Minnesota answers the big questions., title: Agriculture specialist breaks down state of bird flu in Minnesota, link: https://www.yahoo.com/lifestyle/agriculture-specialist-breaks-down-state-155300647.html, date: 2024-06-09T15:54:00+00:00, source: Yahoo', 'snippet: June 3, 2024 — Scientists have discovered some of the molecular mechanisms controlling how plants -- including important crops like soybean and rice -- will respond to rising global temperatures ..., title: Agriculture and Food News, link: https://www.sciencedaily.com/news/plants_animals/agriculture_and_food/, date: 2024-06-07T00:01:00+00:00, source: Science Daily', 'snippet: ICAR-CTCRI launches e-crop, an IoT-based precision agriculture technology, to boost sustainable farming and farmer profits., title: ICAR-CTCRI Launches e-Crop: A Leap in IoT-Based Precision Agriculture, link:

In [ ]:
# Load tools for human interaction (if needed)
human_tools = load_tools(["human"])


In [ ]:
# Define the agriculture news tool
@tool("agriculture news tool")
def agriculture_news_tool(query: str) -> str:
  """Get news for a given agriculture topic."""
  return search_tool.run(query + " agriculture")


In [ ]:
# Initialize the language model
llm = ChatGroq(temperature=0, model_name="llama3-70b-8192")

In [ ]:
# Agent to communicate with the user to find out their interest
customer_communicator = Agent(
    role="Senior agriculture customer communicator",
    goal="Find which agriculture topic the customer is interested in",
    backstory="""You're highly experienced in communicating about agriculture
    and farming technology with customers and their research needs.""",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=human_tools,
)

In [ ]:
# Agent to analyze agriculture news
news_analyst = Agent(
    role="Agriculture News Analyst",
    goal="""Get news for a given agriculture topic. Write 1 paragraph analysis of
    the market and make a prediction - positive, negative, or neutral.""",
    backstory="""You're an expert analyst of trends based on agriculture news.
    You have a complete understanding of macroeconomic factors, but you specialize
    in analyzing news.""",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[agriculture_news_tool],
)


In [ ]:
# Agent to write the final report
writer = Agent(
    role="Agriculture Report Writer",
    goal="""Write a 1 paragraph report of the agriculture market.""",
    backstory="""
    You're widely accepted as the best agriculture analyst that
    understands the market and have tracked every topic for more than 10 years. Your trend
    analysis is always extremely accurate.

    You're also a master-level analyst in traditional markets and have a deep understanding
    of human psychology. You understand macro factors and combine multiple
    theories - e.g. cycle theory. You can hold multiple opinions when analyzing anything.

    You understand news and historical trends, but you look at those with a
    healthy dose of skepticism. You also consider the source of news articles.

    Your most well-developed talent is providing clear and concise summarizations
    that explain very complex market topics in simple terms.

    Some of your writing techniques include:
    - Creating a bullet list (executive summary) of the most important points
    - Distill complex analyses to their most important parts

    Your writing transforms even dry and most technical texts into
    a pleasant and interesting read.""",
    llm=llm,
    verbose=True,
    max_iter=5,
    memory=True,
    allow_delegation=False,
)


In [ ]:
# Define tasks
get_agriculture_topic = Task(
    description="Ask which agriculture topic the customer is interested in.",
    expected_output="Agriculture topic that the human wants you to research.",
    agent=customer_communicator,
)


In [ ]:
get_news_analysis = Task(
    description=f"""
    Use the search tool to get news for the agriculture topic.

    The current date is {datetime.now()}.

    Compose the results into a helpful report""",
    expected_output="""Create a 1 paragraph report for the agriculture topic,
    along with a prediction for the future trend.""",
    agent=news_analyst,
    context=[get_agriculture_topic],
)

In [ ]:
write_report = Task(
    description="""Use the report from the news analyst to
    create a report that summarizes the agriculture topic.""",
    expected_output="""A 1 paragraph report that summarizes the market and
    predicts the future trend for the agriculture topic.""",
    agent=writer,
    context=[get_news_analysis],
)

In [ ]:
# Define the crew with agents and tasks
crew = Crew(
    agents=[customer_communicator, news_analyst, writer],
    tasks=[get_agriculture_topic, get_news_analysis, write_report],
    verbose=2,
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15,
)

# Kickoff the process
results = crew.kickoff()
print(results)



 [DEBUG]: == Working Agent: Senior agriculture customer communicator
 [INFO]: == Starting Task: Ask which agriculture topic the customer is interested in.


> Entering new CrewAgentExecutor chain...
Thought: I need to ask the customer about their interest in agriculture to determine which topic they are interested in.

Action: human
Action Input: {"question": "What aspect of agriculture are you interested in? (e.g. crop management, livestock, precision farming, etc.)"}

What aspect of agriculture are you interested in? (e.g. crop management, livestock, precision farming, etc.)
crop management
 

crop management

Thought: I have received the customer's response, which is crop management.

Final Answer: Crop management.

> Finished chain.
 [DEBUG]: == [Senior agriculture customer communicator] Task output: Crop management.


 [DEBUG]: == Working Agent: Agriculture News Analyst
 [INFO]: == Starting Task: 
    Use the search tool to get news for the agriculture topic.

    The current date